## Object Detection Evaluation

In [ ]:
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve

### IoU (Intersection over Union)
 - measures the overlap between two bounding boxes

In [ ]:
def calculate_iou(pred_box, true_box) -> float:
    """Calculate the Intersection over Union (IoU) of two bounding boxes.

    Args:
        pred_box (list or tuple): Predicted bounding box [x1, y1, x2, y2].
        true_box (list or tuple): Ground truth bounding box [x1, y1, x2, y2].

    Returns:
        float: IoU value.
    """
    x1 = max(pred_box[0], true_box[0])
    y1 = max(pred_box[1], true_box[1])
    x2 = min(pred_box[2], true_box[2])
    y2 = min(pred_box[3], true_box[3])

    intersection = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)
    pred_area = (pred_box[2] - pred_box[0] + 1) * (pred_box[3] - pred_box[1] + 1)
    true_area = (true_box[2] - true_box[0] + 1) * (true_box[3] - true_box[1] + 1)

    union = pred_area + true_area - intersection
    iou = intersection / union
    return iou

### mAP (mean Average Precision)
- comprehensive metric combining precision and recall at a single IoU threshold

In [ ]:
def calc_map(pred_boxes, true_boxes):
    """Calculate the mean Average Precision (mAP) for object detection.
    Args:
        pred_boxes (list of list or tuple): List of predicted bounding boxes [[x1, y1, x2, y2], ...].
        true_boxes (list of list or tuple): List of ground truth bounding boxes [[x1, y1, x2, y2], ...].
    Returns:
        float: mAP value.
    """
    precisions, recalls, _ = precision_recall_curve(true_boxes, pred_boxes)
    map_score = average_precision_score(true_boxes, pred_boxes)
    return map_score

### Evaluation of the YOLOv5 model

In [ ]:
model = torch.hub.load('ultralytics/yolov5',
                       'yolov5s',
                       pretrained=True)

loader = torch.utils.data.DataLoader()

model.eval()
with torch.no_grad():
    for images, targets in loader:
        predictions = model(images)
        iou = calculate_iou(predictions['boxes'], targets['boxes'])
        map_score = calc_map(predictions['boxes'], targets['boxes'])
        print(f'IoU: {iou}, MAP: {map_score}')